In [ ]:
code = 'StraddleSELL_LaterHedgeBRE'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def HedgeBRE(bt, start_time, end_time, method, hedge_premium_percent, hedge_sl, hedge_decay):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        
        from_candle_close = True if method == 'CC' else False

        while True:

            sell_ce_scrip, sell_pe_scrip, sell_ce_price, sell_pe_price, future_price, sell_start_dt = bt.get_strike(start_dt, end_dt, om=0)
            if sell_ce_scrip is None: return None

            straddle_premium_avg = (sell_ce_price+sell_pe_price)/2
            hedge_premium = cal_percent(straddle_premium_avg, hedge_premium_percent)

            hdg_ce_scrip, hdg_pe_scrip, hdg_ce_price, hdg_pe_price, _, hdg_start_dt = bt.get_strike(sell_start_dt, end_dt, target=hedge_premium, obove_target_only=True)
            if hdg_ce_scrip is None: return None
            
            if sell_start_dt == hdg_start_dt:
                start_dt = sell_start_dt
                break
            else:
                start_dt = sell_start_dt + datetime.timedelta(minutes=1)

        entry_time = start_dt
        std_open, std_high, std_low, std_close, _, _, _, _, _, _, std_pnl = bt.sl_check_combine_leg(start_dt, end_dt, sell_ce_scrip, sell_pe_scrip, orderside='SELL', with_ohlc=True)

        first_straddle = [f"({sell_ce_scrip}, {sell_pe_scrip})", std_open, std_close, std_pnl]
        
        #HEDGE BUY Target
        ce_hedge_buy_target = hdg_ce_price + cal_percent(hdg_ce_price, hedge_decay)
        pe_hedge_buy_target = hdg_pe_price + cal_percent(hdg_pe_price, hedge_decay)

        # check entry
        hdg_ce_open, hdg_ce_high, hdg_ce_low, hdg_ce_close, hdg_ce_decay_price, hdg_ce_decay_flag, hdg_ce_decay_time = bt.decay_check_single_leg(start_dt, end_dt, hdg_ce_scrip, decay_price=ce_hedge_buy_target, from_candle_close=from_candle_close, orderside='BUY', with_ohlc=True)
        hdg_pe_open, hdg_pe_high, hdg_pe_low, hdg_pe_close, hdg_pe_decay_price, hdg_pe_decay_flag, hdg_pe_decay_time = bt.decay_check_single_leg(start_dt, end_dt, hdg_pe_scrip, decay_price=pe_hedge_buy_target, from_candle_close=from_candle_close, orderside='BUY', with_ohlc=True)
        
        # check sl
        hdg_ce_sl_price, hdg_ce_sl_flag, _, _, hdg_ce_sl_time, hdg_ce_pnl = bt.sl_check_single_leg(hdg_ce_decay_time, end_dt, hdg_ce_scrip, o=(None if method == 'CC' else hdg_ce_decay_price), sl=hedge_sl, orderside='BUY', from_candle_close=from_candle_close)
        hdg_pe_sl_price, hdg_pe_sl_flag, _, _, hdg_pe_sl_time, hdg_pe_pnl = bt.sl_check_single_leg(hdg_pe_decay_time, end_dt, hdg_pe_scrip, o=(None if method == 'CC' else hdg_pe_decay_price), sl=hedge_sl, orderside='BUY', from_candle_close=from_candle_close)

        hdg_ce_first_entrie = [hdg_ce_scrip, hdg_ce_open, hdg_ce_high, hdg_ce_low, hdg_ce_close, hdg_ce_decay_price, hdg_ce_decay_flag, hdg_ce_decay_time, hdg_ce_sl_price, hdg_ce_sl_flag, hdg_ce_sl_time, hdg_ce_pnl]
        hdg_pe_first_entrie = [hdg_pe_scrip, hdg_pe_open, hdg_pe_high, hdg_pe_low, hdg_pe_close, hdg_pe_decay_price, hdg_pe_decay_flag, hdg_pe_decay_time, hdg_pe_sl_price, hdg_pe_sl_flag, hdg_pe_sl_time, hdg_pe_pnl]
        
        hdg_ce_re_entries = []
        for re_no in range(max_re):
            if hdg_ce_sl_time and re_no < re_entries:
                start_dt = hdg_ce_sl_time
                _, hdg_ce_decay_flag, hdg_ce_decay_time = bt.decay_check_single_leg(start_dt, end_dt, hdg_ce_scrip, decay_price=ce_hedge_buy_target, from_candle_close=from_candle_close, orderside='BUY')

                if hdg_ce_decay_flag:
                    _, hdg_ce_sl_flag, _, _, hdg_ce_sl_time, hdg_ce_pnl = bt.sl_check_single_leg(hdg_ce_decay_time, end_dt, hdg_ce_scrip, o=(None if method == 'CC' else hdg_ce_decay_price), sl=hedge_sl, orderside='BUY', from_candle_close=from_candle_close)
                else:
                    hdg_ce_sl_flag, hdg_ce_sl_time, hdg_ce_pnl = False, '', 0

                hdg_ce_re_entries.extend([hdg_ce_decay_flag, hdg_ce_decay_time, hdg_ce_sl_flag, hdg_ce_sl_time, hdg_ce_pnl])
            else:
                hdg_ce_re_entries.extend([False, '', False, '', 0])

        hdg_pe_re_entries = []
        for re_no in range(max_re):
            if hdg_pe_sl_time and re_no < re_entries:
                start_dt = hdg_pe_sl_time
                _, hdg_pe_decay_flag, hdg_pe_decay_time = bt.decay_check_single_leg(start_dt, end_dt, hdg_pe_scrip, decay_price=pe_hedge_buy_target, from_candle_close=from_candle_close, orderside='BUY')

                if hdg_pe_decay_flag:
                    _, hdg_pe_sl_flag, _, _, hdg_pe_sl_time, hdg_pe_pnl = bt.sl_check_single_leg(hdg_pe_decay_time, end_dt, hdg_pe_scrip, o=(None if method == 'CC' else hdg_pe_decay_price), sl=hedge_sl, orderside='BUY', from_candle_close=from_candle_close)
                else:
                    hdg_pe_sl_flag, hdg_pe_sl_time, hdg_pe_pnl = False, '', 0

                hdg_pe_re_entries.extend([hdg_pe_decay_flag, hdg_pe_decay_time, hdg_pe_sl_flag, hdg_pe_sl_time, hdg_pe_pnl])
            else:
                hdg_pe_re_entries.extend([False, '', False, '', 0])
            
        return [code, bt.index, start_time, end_time, method, hedge_premium_percent, hedge_sl, hedge_decay, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + first_straddle + hdg_ce_first_entrie + hdg_ce_re_entries + hdg_pe_first_entrie + hdg_pe_re_entries
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, method, hedge_premium_percent, hedge_sl, hedge_decay])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re, re_entries = 7, 7

            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_Method/P_HedgePremium%/P_Hedge_SL/P_HedgeDecay/Date/Day/DTE/EntryTime/Future/STD.Scrip/STD.Open/STD.Close/STD.PNL')

            log_cols += '/CE.Hdg.Scrip/CE.Hdg.Open/CE.Hdg.High/CE.Hdg.Low/CE.Hdg.Close/CE.Hdg.Buy.Price/CE.Hdg.Buy.Flag/CE.Hdg.Buy.Time/CE.Hdg.SL.Price/CE.Hdg.SL.Flag/CE.Hdg.SL.Time/CE0.Hdg.PNL'
            for re in range(1, max_re+1):
                log_cols += f'/CE{re}.Hdg.Buy.Flag/CE{re}.Hdg.Buy.Time/CE{re}.Hdg.SL.Flag/CE{re}.Hdg.SL.Time/CE{re}.Hdg.PNL'
            
            log_cols += '/PE.Hdg.Scrip/PE.Hdg.Open/PE.Hdg.High/PE.Hdg.Low/PE.Hdg.Close/PE.Hdg.Buy.Price/PE.Hdg.Buy.Flag/PE.Hdg.Buy.Time/PE.Hdg.SL.Price/PE.Hdg.SL.Flag/PE.Hdg.SL.Time/PE0.Hdg.PNL'
            for re in range(1, max_re+1):
                log_cols += f'/PE{re}.Hdg.Buy.Flag/PE{re}.Hdg.Buy.Time/PE{re}.Hdg.SL.Flag/PE{re}.Hdg.SL.Time/PE{re}.Hdg.PNL'
            
            log_cols = log_cols.split('/')
            
            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [HedgeBRE(bt, row.entry_time, row.exit_time, row.method, row.hedge_premium_pct, row.hedge_sl, row.hedge_decay) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))